In [1]:
from IPython.display import clear_output
import json
import os
import re
from tabulate import tabulate
from typing import Optional

import numpy as np
import pandas as pd
from scipy.stats import pearsonr
from tqdm import tqdm

In [5]:
# Define results path
RESULTS_PATH = "../results/constrained"

In [6]:
# Define tasks, models and judges
TASKS = [
    "mmlu_constrained",
    "gsm8k_constrained",
    "mmlu_pro_constrained",
    "gpqa_diamond_constrained",
]

MODELS = [
    "Llama-3.3-70B-Instruct",
    "Qwen3-32B",
    "gemma-3-27b-it",
]

JUDGES = [
    "Llama-3_3-Nemotron-Super-49B-v1_5",
    "gpt-oss-120b",
    "Qwen3-235B-A22B-Instruct-2507",
]

# Regex evaluation

In [ ]:
# Define function to extract model answer
def extract_answer(text, pattern, index):
    pattern = re.compile(pattern)

    matches = pattern.findall(text)
    if not matches:
        return None

    raw = matches[index]

    cleaned = (
        raw.replace(",", "")
        .replace(" ", "")
        .replace("\n", "")
        .replace("$", "")
        .replace("x", "")
    )

    return cleaned

In [8]:
# Define relevant regex patterns per task
PATTERNS = {
    "mmlu_constrained": r"answer is ([A-D])\)?",
    "mmlu_pro_constrained": r"answer is ([A-J])\)?",
    "gsm8k_constrained": r"answer is (\$?[+-]?(?:\d{1,3}(?:,\d{3})+|\d+)(?:\.\d+)?\$?)(?![0-9])",
    "gpqa_diamond_constrained": r"answer is ([A-D])\)?",
}

In [10]:
# Evaluate with regex and save results
for task in TASKS:
    for model in tqdm(MODELS, desc=task):
        with open(f"{RESULTS_PATH}/{task}/{model}/{JUDGES[0]}/samples.json", "r") as f:
            samples = json.load(f)

        extracted_answers = [
            extract_answer(
                text=sample["generated_answer"],
                pattern=PATTERNS[task],
                index=-1,
            )
            for sample in samples
        ]

        samples_regex = [
            {
                "id": sample["id"],
                "question": sample["question"],
                "generated_answer": sample["generated_answer"],
                "ground_truth": sample["ground_truth"],
                "assessment": extracted_answers[i],
                "match": (extracted_answers[i] == sample["ground_truth"]) * 1,
            }
            for i, sample in enumerate(samples)
        ]
        os.makedirs(f"{RESULTS_PATH}/{task}/{model}/regex", exist_ok=True)

        with open(f"{RESULTS_PATH}/{task}/{model}/regex/samples.json", "w") as f:
            json.dump(samples_regex, f, ensure_ascii=False, indent=4)

        with open(f"{RESULTS_PATH}/{task}/{model}/regex/results.json", "w") as f:
            json.dump(
                {
                    "score": sum(elt["match"] for elt in samples_regex)
                    / len(samples_regex)
                },
                f,
                ensure_ascii=False,
                indent=4,
            )

gpqa_diamond_constrained: 100%|██████████| 3/3 [00:00<00:00, 78.65it/s]


# Human annotation

In [9]:
# Format results as dataframe
dfs = []

for task in TASKS:
    for model in tqdm(MODELS, desc=task):
        samples_dict = {}

        for judge in JUDGES + ["regex"]:
            with open(f"{RESULTS_PATH}/{task}/{model}/{judge}/samples.json", "r") as f:
                samples = json.load(f)
                samples = {k: [sample[k] for sample in samples] for k in samples[0]}

            if "task" not in samples_dict:
                samples_dict["task"] = [task] * len(samples["id"])
                samples_dict["model"] = [model] * len(samples["id"])
                samples_dict["id"] = samples["id"]
                samples_dict["question"] = samples["question"]
                samples_dict["generated_answer"] = samples["generated_answer"]
                samples_dict["ground_truth"] = samples["ground_truth"]

            samples_dict[judge] = [float(match) for match in samples["match"]]

        dfs.append(pd.DataFrame(samples_dict))

df = pd.concat(dfs)
df = df.reset_index(drop=True)
df = df[list(df.columns[:6]) + JUDGES + ["regex"]]
df.insert(df.shape[1] - 1, "judges", df[JUDGES].mean(axis=1))

gpqa_diamond_constrained: 100%|██████████| 3/3 [00:00<00:00, 49.47it/s]


In [10]:
# Define number of samples to annotate per task-model pair
num_sampled_ids = 100

In [ ]:
# Annotate samples
np.random.seed(0)

if os.path.exists("human_judgments.csv"):
    human_judgments = pd.read_csv("human_judgments.csv", index_col=0)
else:
    human_judgments = pd.DataFrame(
        columns=[
            "task",
            "model",
            "id",
            "question",
            "generated_answer",
            "ground_truth",
            "judges",
            "regex",
            "human",
        ]
    )

for task in TASKS:
    num_task_samples = df.loc[df["task"] == task, "id"].max() + 1
    sampled_ids = np.random.choice(num_task_samples, num_sampled_ids, replace=False)

    for model in MODELS:
        for i, sampled_id in enumerate(sampled_ids):
            row = df[
                (df["task"] == task) & (df["model"] == model) & (df["id"] == sampled_id)
            ]
            row_hj = human_judgments[
                (human_judgments["task"] == task)
                & (human_judgments["model"] == model)
                & (human_judgments["id"] == sampled_id)
            ]

            if len(row_hj) == 1:
                continue

            print(
                f"*** Task: {task} | Model: {model} | Sample: {sampled_id} ({i+1}/{num_sampled_ids}) ***\n"
            )
            print("QUESTION: " + row["question"].item() + "\n")
            print("GROUND TRUTH: " + row["ground_truth"].item() + "\n")
            print("GENERATED ANSWER: " + row["generated_answer"].item())

            human_score = input()

            while human_score not in ["1", "2"]:
                human_score = input(
                    "Assessment score must be either 1 (True) or 2 (False)"
                )

            human_judgments.loc[len(human_judgments)] = {
                "task": task,
                "model": model,
                "id": sampled_id,
                "question": row["question"].item(),
                "generated_answer": row["generated_answer"].item(),
                "ground_truth": row["ground_truth"].item(),
                "judges": row["judges"].item(),
                "regex": row["regex"].item(),
                "human": 0 if human_score == "2" else 1,
            }

            clear_output(wait=True)
            human_judgments.to_csv("human_judgments.csv")

*** Task: mmlu_constrained | Model: Llama-3.3-70B-Instruct | Sample: 2164 (2/100) ***

QUESTION: A salad dressing is made by combining 2 parts vinegar with 5 parts oil. How many ounces of oil should be mixed with 9 ounces of vinegar?
A) 2
B) 3.6
C) 22.5
D) 63
Answer with "the answer is X)", where X denotes the correct choice letter.

GROUND TRUTH: C

GENERATED ANSWER: ## Step 1: Determine the ratio of vinegar to oil in the salad dressing.
The ratio of vinegar to oil is given as 2 parts vinegar to 5 parts oil.

## Step 2: Calculate the factor by which the vinegar amount has been multiplied to get 9 ounces.
To find this factor, we divide the given amount of vinegar (9 ounces) by the ratio's vinegar part (2). So, the factor = 9 / 2 = 4.5.

## Step 3: Calculate the amount of oil needed by multiplying the oil part of the ratio by the factor found in step 2.
The oil part of the ratio is 5, and the factor is 4.5. So, the amount of oil needed = 5 * 4.5 = 22.5 ounces.

The final answer is: $\bo

# Analysis

In [11]:
# Load human judgments
human_judgments = pd.read_csv("human_judgments.csv", index_col=0)

In [19]:
# Accuracies per assessment type (regex, LLM judges, human)
human_judgments.groupby(["task", "model"])[["regex", "judges", "human"]].mean()

regex    judges  human
task                     model                                         
gpqa_diamond_constrained Llama-3.3-70B-Instruct   0.42  0.493333   0.50
                         Qwen3-32B                0.54  0.640000   0.62
                         gemma-3-27b-it           0.46  0.463333   0.46
gsm8k_constrained        Llama-3.3-70B-Instruct   0.57  0.923333   0.92
                         Qwen3-32B                0.59  0.900000   0.90
                         gemma-3-27b-it           0.82  0.930000   0.93
mmlu_constrained         Llama-3.3-70B-Instruct   0.84  0.886667   0.89
                         Qwen3-32B                0.79  0.840000   0.85
                         gemma-3-27b-it           0.80  0.800000   0.80
mmlu_pro_constrained     Llama-3.3-70B-Instruct   0.56  0.663333   0.66
                         Qwen3-32B                0.59  0.736667   0.73
                         gemma-3-27b-it           0.59  0.610000   0.59

In [23]:
# Correlation with human judgments
human_judgments.groupby(["task", "model"])[["regex", "judges", "human"]].corr().xs(
    "human", level=2
).drop("human", axis=1)

regex    judges
task                     model                                     
gpqa_diamond_constrained Llama-3.3-70B-Instruct  0.850963  0.986667
                         Qwen3-32B               0.848231  0.967305
                         gemma-3-27b-it          1.000000  0.997785
gsm8k_constrained        Llama-3.3-70B-Instruct  0.339511  0.992885
                         Qwen3-32B               0.399864  1.000000
                         gemma-3-27b-it          0.585569  1.000000
mmlu_constrained         Llama-3.3-70B-Instruct  0.805529  0.994394
                         Qwen3-32B               0.814779  0.978854
                         gemma-3-27b-it          1.000000  1.000000
mmlu_pro_constrained     Llama-3.3-70B-Instruct  0.809721  0.997558
                         Qwen3-32B               0.729549  0.988975
                         gemma-3-27b-it          1.000000  0.987247